# 02 — 結果集計 & 学習曲線プロット

Driveに蓄積した `results/*.json` を読み込み、訓練サイズ vs MRE/SDR をプロットする。
全ジョブが揃っていなくても途中経過として実行できる。

In [ ]:
DRIVE_LC_DIR = "/content/drive/MyDrive/anglist_learning_curve"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt

results_dir = os.path.join(DRIVE_LC_DIR, "results")
all_results = []
for fname in sorted(os.listdir(results_dir)):
    if fname.endswith(".json"):
        with open(os.path.join(results_dir, fname)) as f:
            all_results.append(json.load(f))

print(f"読み込み済みジョブ数: {len(all_results)} / 30")

In [ ]:
# サイズ別に集計（複数フォールドの平均 ± std）
from collections import defaultdict

size_data = defaultdict(lambda: {"mre": [], "sdr2": [], "sdr4": []})

for r in all_results:
    s = r["size"]
    size_data[s]["mre"].append(r["overall"]["mre_mm"])
    size_data[s]["sdr2"].append(r["overall"]["sdr2"])
    size_data[s]["sdr4"].append(r["overall"]["sdr4"])

sizes = sorted(size_data.keys())
mre_mean  = [np.mean(size_data[s]["mre"])  for s in sizes]
mre_std   = [np.std(size_data[s]["mre"])   for s in sizes]
sdr2_mean = [np.mean(size_data[s]["sdr2"]) for s in sizes]
sdr2_std  = [np.std(size_data[s]["sdr2"])  for s in sizes]
sdr4_mean = [np.mean(size_data[s]["sdr4"]) for s in sizes]
sdr4_std  = [np.std(size_data[s]["sdr4"])  for s in sizes]

print(f"{'Size':>6}  {'MRE(mm)':>10}  {'SDR@2mm':>10}  {'SDR@4mm':>10}  {'N folds':>8}")
for i, s in enumerate(sizes):
    n = len(size_data[s]["mre"])
    print(f"{s:>6}  {mre_mean[i]:>6.2f}±{mre_std[i]:.2f}  {sdr2_mean[i]:>6.1f}±{sdr2_std[i]:.1f}%  {sdr4_mean[i]:>6.1f}±{sdr4_std[i]:.1f}%  {n:>8}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Learning Curve (5-fold CV)", fontsize=14)

def plot_metric(ax, mean, std, ylabel, title, color):
    ax.plot(sizes, mean, "o-", color=color)
    ax.fill_between(sizes,
                    [m - s for m, s in zip(mean, std)],
                    [m + s for m, s in zip(mean, std)],
                    alpha=0.2, color=color)
    ax.set_xlabel("Training samples")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(sizes)

plot_metric(axes[0], mre_mean,  mre_std,  "MRE (mm)",   "MRE vs Training Size",     "steelblue")
plot_metric(axes[1], sdr2_mean, sdr2_std, "SDR@2mm (%)","SDR@2mm vs Training Size", "darkorange")
plot_metric(axes[2], sdr4_mean, sdr4_std, "SDR@4mm (%)","SDR@4mm vs Training Size", "seagreen")

plt.tight_layout()
out_path = os.path.join(DRIVE_LC_DIR, "learning_curve.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"保存: {out_path}")

In [ ]:
# ランドマーク別MRE（全ジョブ揃った場合）
LANDMARK_ORDER = ["L1_ant", "L1_post", "S1_ant", "S1_post", "FH"]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["steelblue", "darkorange", "seagreen", "crimson", "purple"]

for lm, color in zip(LANDMARK_ORDER, colors):
    lm_size_mre = defaultdict(list)
    for r in all_results:
        if lm in r["landmarks"]:
            lm_size_mre[r["size"]].append(r["landmarks"][lm]["mre_mm"])
    lm_sizes = sorted(lm_size_mre.keys())
    lm_mean = [np.mean(lm_size_mre[s]) for s in lm_sizes]
    ax.plot(lm_sizes, lm_mean, "o-", label=lm, color=color)

ax.set_xlabel("Training samples")
ax.set_ylabel("MRE (mm)")
ax.set_title("Per-landmark MRE vs Training Size")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(sizes)
plt.tight_layout()
out_path2 = os.path.join(DRIVE_LC_DIR, "learning_curve_per_landmark.png")
plt.savefig(out_path2, dpi=150, bbox_inches="tight")
plt.show()
print(f"保存: {out_path2}")